In [21]:
import pandas as pd
import os

In [25]:
def load_naics_descriptions_from_excel(excel_file):
    # Load the NAICS descriptions from an Excel sheet
    df = pd.read_excel(excel_file)
    
    # Clean column names to avoid issues with leading/trailing spaces
    df.columns = df.columns.str.strip()

    # Check if the required columns are present
    if 'NAICS Code' not in df.columns or 'NAICS Description' not in df.columns:
        print(f"Available columns in the file: {df.columns.tolist()}")
        raise KeyError("The required columns 'NAICS Code' and/or 'NAICS Description' are not found in the file.")
    
    # Create a dictionary from the DataFrame
    naics_dict = pd.Series(df['NAICS Description'].values, index=df['NAICS Code'].astype(str)).to_dict()
    return naics_dict

In [26]:
def generate_naics_descriptions(input_file, naics_excel_file):
    # Load NAICS descriptions dictionary from the Excel file
    naics_descriptions = load_naics_descriptions_from_excel(naics_excel_file)

    # Read the input Excel file
    df = pd.read_excel(input_file)

    # Function to get NAICS descriptions
    def get_naics_description(codes):
        # Remove spaces from the codes
        cleaned_codes = codes.replace(' ', '')
        code_list = cleaned_codes.split(';')
        descriptions = [
            f"{code.strip()}: {naics_descriptions[code.strip()]}"
            for code in code_list if code.strip() in naics_descriptions
        ]
        return '; '.join(descriptions)

    # Apply the conversion to the 'NAICS Found' column
    df['NAICS Found'] = df['NAICS Found'].astype(str).apply(get_naics_description)

    # Remove rows where 'NAICS Found' is empty after filtering
    df = df[df['NAICS Found'] != '']

    # Generate the output file name
    output_file = os.path.splitext(input_file)[0] + '_output.xlsx'

    # Write the output to a new Excel file
    df.to_excel(output_file, index=False)

    print(f"Output file generated: {output_file}")

In [27]:
if __name__ == "__main__":
    # Prompt the user for file names
    input_file = input("Enter the name of the input Excel file (with extension): ")
    naics_excel_file = input("Enter the name of the NAICS descriptions Excel file (with extension): ")

    # Assuming the files are in the current working directory
    current_directory = os.getcwd()
    input_file_path = os.path.join(current_directory, input_file)
    naics_excel_file_path = os.path.join(current_directory, naics_excel_file)

    generate_naics_descriptions(input_file_path, naics_excel_file_path)

Enter the name of the input Excel file (with extension): Spend by Supplier - Trailing 12 Months (1) (1).xlsx
Enter the name of the NAICS descriptions Excel file (with extension): Naics description sheet.xlsx
Output file generated: /Users/louis.standridge/Desktop/Spend by Supplier - Trailing 12 Months (1) (1)_output.xlsx
